# SVR-based MRI Denoising

This notebook investigates classical supervised MRI denoising using
Support Vector Regression (SVR).

The method learns a mapping between noisy image patches and their corresponding clean central pixels. Several experimental settings are evaluated, including different noise types, noise levels, patch sizes, and patch extraction strategies.

#### Experimental setup

The SVR experiments use fixed synthetically corrupted MRI slices generated in "02_synthetic_noise.ipynb". Unlike the deep-learning experiments, noise is not generated on-the-fly. This ensures that all SVR configurations are evaluated using the same corrupted observations.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import utils, noise

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
)

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "raw_data_split"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

In [4]:
def get_nii_files(directory):
    return sorted(directory.glob("*.nii.gz"))

train_files = get_nii_files(TRAIN_DIR)
val_files   = get_nii_files(VAL_DIR)
test_files  = get_nii_files(TEST_DIR)

print(f"Train patients : {len(train_files)}")
print(f"Val patients   : {len(val_files)}")
print(f"Test patients  : {len(test_files)}")

Train patients : 412
Val patients   : 51
Test patients  : 52


### Patch-based SVR denoising

In [5]:
SEED = 42
rng = np.random.default_rng(SEED)

N_TRAIN_PATIENTS = 20
N_VAL_PATIENTS   = 5
N_TEST_PATIENTS  = 5

svr_train_files = sorted(
    rng.choice(train_files, N_TRAIN_PATIENTS, replace=False)
)

svr_val_files = sorted(
    rng.choice(val_files, N_VAL_PATIENTS, replace=False)
)

svr_test_files = sorted(
    rng.choice(test_files, N_TEST_PATIENTS, replace=False)
)

print(f"SVR train : {len(svr_train_files)} patients")
print(f"SVR val   : {len(svr_val_files)} patients")
print(f"SVR test  : {len(svr_test_files)} patients")

SVR train : 20 patients
SVR val   : 5 patients
SVR test  : 5 patients


In [7]:
SLICES_PER_TRAIN_PATIENT = 3

def sample_slices(files, n_slices, seed=42):
    rng = np.random.default_rng(seed)
    samples = []

    for path in files:
        volume = utils.lire_nii(path)

        n = min(n_slices, volume.shape[0])

        indices = rng.choice(
            volume.shape[0],
            size=n,
            replace=False
        )

        for idx in sorted(indices):
            samples.append((path, int(idx)))

    return samples


train_samples = sample_slices(
    svr_train_files,
    SLICES_PER_TRAIN_PATIENT,
    seed=SEED
)

print("Training slices selected:", len(train_samples))

Training slices selected: 60


In [ ]:
from src import utils, noise
import importlib

importlib.reload(utils)
importlib.reload(noise)


NOISE_CONFIG = {
    "low": {
        "gaussian": 30,
        "rician": 40,
        "mixed": (25, 30),
    },
    "medium": {
        "gaussian": 40,
        "rician": 53,
        "mixed": (33, 40),
    },
    "strong": {
        "gaussian": 80,
        "rician": 50,
        "mixed": (60, 40),
    },
}



# Reconstruction

def mirror_pad(img, pad_size):
    return np.pad(
        img,
        pad_width=pad_size,
        mode="reflect"
    )


def reconstruct_image_svr(model, scaler, noisy_img, patch_size=5):
    offset = patch_size // 2
    padded = np.pad(noisy_img, offset, mode="reflect")

    h, w = noisy_img.shape
    patches = np.empty(
        (h * w, patch_size * patch_size),
        dtype=np.float32
    )

    k = 0
    for i in range(h):
        for j in range(w):
            patches[k] = padded[
                i:i + patch_size,
                j:j + patch_size
            ].ravel()
            k += 1

    patches = scaler.transform(patches)
    predictions = model.predict(patches)

    return predictions.reshape(h, w)


from src import metrics 
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def evaluate_denoising(clean, noisy, denoised, data_range=4095):

    return {
        "PSNR noisy": peak_signal_noise_ratio(
            clean, noisy,
            data_range=data_range
        ),

        "PSNR denoised": peak_signal_noise_ratio(
            clean, denoised,
            data_range=data_range
        ),

        "SSIM noisy": structural_similarity(
            clean, noisy,
            data_range=data_range
        ),

        "SSIM denoised": structural_similarity(
            clean, denoised,
            data_range=data_range
        ),

        "AKLD": metrics.akld_image(
            noisy,
            denoised,
            clean
        ),

        "Blur clean": metrics.blur_metric(clean),

        "Blur noisy": metrics.blur_metric(noisy),

        "Blur denoised": metrics.blur_metric(denoised)
    }



from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

def train_svr_configuration(
    train_samples,
    noise_level,
    patch_size,
    stride,
    noise_types=("gaussian", "rician", "mixed"),
    max_samples=300_000,
    seed=42
):
    X_total = []
    y_total = []

    for sample_id, (path, slice_idx) in enumerate(train_samples):

        volume = utils.lire_nii(path)
        clean = volume[slice_idx]

        for noise_id, noise_type in enumerate(noise_types):

            noisy = noise.apply_noise(
                clean,
                noise_type=noise_type,
                level=noise_level,
                noise_config=NOISE_CONFIG,
                seed=seed + sample_id * len(noise_types) + noise_id
            )

            X, y = extract_patches(
                noisy,
                clean,
                patch_size=patch_size,
                stride=stride
            )

            X_total.append(X)
            y_total.append(y)

    X_total = np.concatenate(X_total)
    y_total = np.concatenate(y_total)

    # Reproducible subsampling
    rng = np.random.default_rng(seed)

    if len(X_total) > max_samples:
        idx = rng.choice(
            len(X_total),
            size=max_samples,
            replace=False
        )

        X_total = X_total[idx]
        y_total = y_total[idx]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_total)

    model = SVR(
        kernel="rbf",
        C=10,
        epsilon=0.1,
        gamma="scale"
    )

    model.fit(X_scaled, y_total)

    return model, scaler



### I. Low-level Noise

In [ ]:
# Train data
noise_types = ("gaussian", "rician", "mixed")

patch_size = 5
stride = 3

X_total = []
y_total = []

for sample_id, (path, slice_idx) in enumerate(train_samples):

    volume = utils.lire_nii(path)
    clean = volume[slice_idx]

    noisy = noise.apply_noise(
        clean,
        NOISE_CONFIG,
        noise_type="gaussian",
        level="low",
        seed=SEED + sample_id
    )

    X, y = utils.extract_patches(
        noisy,
        clean,
        patch_size=patch_size,
        stride=stride
    )

    X_total.append(X)
    y_total.append(y)

X_total = np.concatenate(X_total)
y_total = np.concatenate(y_total)

print("X_total :", X_total.shape)
print("y_total :", y_total.shape)

X_total : (688839, 25)
y_total : (688839,)


In [32]:
MAX_TRAIN_SAMPLES = 300_000

rng = np.random.default_rng(SEED)

indices = rng.choice(
    len(X_total),
    size=min(MAX_TRAIN_SAMPLES, len(X_total)),
    replace=False
)

X_train = X_total[indices]
y_train = y_total[indices]

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)

X_train : (300000, 25)
y_train : (300000,)


In [ ]:
# Model & training

PATCH_SIZES = [3, 5, 7, 9]

patch_results = {}

for patch_size in PATCH_SIZES:

    # Historical overlapping configuration
    stride = (patch_size + 1) // 2

    model, scaler = train_svr_configuration(
        train_samples=train_samples,
        noise_level="low",
        patch_size=patch_size,
        stride=stride,
        noise_types=("gaussian", "rician", "mixed"),
        max_samples=300_000,
        seed=SEED
    )

    patch_results[patch_size] = {
        "model": model,
        "scaler": scaler,
        "stride": stride
    }

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",10
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [ ]:
# Fixed validation image for patch-size comparison
val_path = svr_val_files[0]

volume = utils.lire_nii(val_path)
slice_idx = volume.shape[0] // 2

clean_val = volume[slice_idx]

# Apply low noise to the clean validation image
noisy_val = noise.apply_noise(
    clean_val,
    noise_type="mixed",
    level="low",
    noise_config=NOISE_CONFIG,
    seed=SEED
)


# evaluation 

patch_metrics = []

for patch_size, config in patch_results.items():

    print(f"Evaluating patch size {patch_size}x{patch_size}...")

    denoised = reconstruct_image_svr(
        model=config["model"],
        scaler=config["scaler"],
        noisy_img=noisy_val,
        patch_size=patch_size
    )

    metrics_result = evaluate_denoising(
        clean_val,
        noisy_val,
        denoised
    )

    metrics_result["patch_size"] = patch_size
    metrics_result["noise_type"] = noise_types
    metrics_result["noise_level"] = "low"

In [ ]:
import pandas as pd

df_patch_results = pd.DataFrame(patch_metrics)

df_patch_results

### II. Medium-level Noise

In [ ]:
PATCH_SIZES = [3, 5, 7, 9]

patch_results_medium = {}

for patch_size in PATCH_SIZES:

    stride = (patch_size + 1) // 2

    model, scaler = train_svr_configuration(
        train_samples=train_samples,
        noise_level="medium",
        patch_size=patch_size,
        stride=stride,
        noise_types=("gaussian", "rician", "mixed"),
        max_samples=300_000,
        seed=SEED
    )

    patch_results_medium[patch_size] = {
        "model": model,
        "scaler": scaler,
        "stride": stride
    }

In [ ]:
val_path = svr_val_files[0]

volume = utils.lire_nii(val_path)
slice_idx = volume.shape[0] // 2

clean_val = volume[slice_idx]

noisy_val_medium = noise.apply_noise(
    clean_val,
    noise_type="mixed",
    level="medium",
    noise_config=NOISE_CONFIG,
    seed=SEED
)




# Eval

patch_metrics_medium = []

for patch_size, config in patch_results_medium.items():

    print(f"Evaluating patch size {patch_size}x{patch_size}...")

    denoised = reconstruct_image_svr(
        model=config["model"],
        scaler=config["scaler"],
        noisy_img=noisy_val_medium,
        patch_size=patch_size
    )

    metrics_result = evaluate_denoising(
        clean_val,
        noisy_val_medium,
        denoised
    )

    metrics_result["patch_size"] = patch_size
    metrics_result["stride"] = config["stride"]
    metrics_result["noise_type"] = noise_types
    metrics_result["noise_level"] = "medium"

    patch_metrics_medium.append(metrics_result)

In [ ]:
df_patch_results_medium = pd.DataFrame(patch_metrics_medium)

df_patch_results_medium

### III. Strong-level Noise

In [ ]:
patch_results_strong = {}

for patch_size in PATCH_SIZES:

    stride = (patch_size + 1) // 2

    model, scaler = train_svr_configuration(
        train_samples=train_samples,
        noise_level="strong",
        patch_size=patch_size,
        stride=stride,
        noise_types=("gaussian", "rician", "mixed"),
        max_samples=300_000,
        seed=SEED
    )

    patch_results_strong[patch_size] = {
        "model": model,
        "scaler": scaler,
        "stride": stride
    }

In [ ]:
noisy_val_strong = noise.apply_noise(
    clean_val,
    noise_type="mixed",
    level="strong",
    noise_config=NOISE_CONFIG,
    seed=SEED
)

patch_metrics_strong = []

for patch_size, config in patch_results_strong.items():

    print(f"Evaluating patch size {patch_size}x{patch_size}...")

    denoised = reconstruct_image_svr(
        model=config["model"],
        scaler=config["scaler"],
        noisy_img=noisy_val_strong,
        patch_size=patch_size
    )

    metrics_result = evaluate_denoising(
        clean_val,
        noisy_val_strong,
        denoised
    )

    metrics_result["patch_size"] = patch_size
    metrics_result["stride"] = config["stride"]
    metrics_result["noise_type"] = noise_types
    metrics_result["noise_level"] = "strong"

    patch_metrics_strong.append(metrics_result)

df_patch_results_strong = pd.DataFrame(patch_metrics_strong)

df_patch_results_strong

In [ ]:
df_noise_levels = pd.concat([
    df_patch_results,          # low
    df_patch_results_medium,   # medium
    df_patch_results_strong    # strong
], ignore_index=True)

df_noise_levels

#### Generalization 

In [ ]:
# Gaussian only

patch_size = 5
config = patch_results_medium[patch_size]

noisy_gaussian = noise.apply_noise(
    clean_val,
    noise_type="gaussian",
    level="medium",
    noise_config=NOISE_CONFIG,
    seed=SEED
)

denoised_gaussian = reconstruct_image_svr(
    model=config["model"],
    scaler=config["scaler"],
    noisy_img=noisy_gaussian,
    patch_size=patch_size
)

results_gaussian = evaluate_denoising(
    clean_val,
    noisy_gaussian,
    denoised_gaussian
)

print(results_gaussian)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(clean_val, cmap="gray")
plt.title("Clean")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(noisy_gaussian, cmap="gray")
plt.title("Gaussian medium")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(denoised_gaussian, cmap="gray")
plt.title(f"SVR denoised ({patch_size}x{patch_size})")
plt.axis("off")

plt.show()

In [ ]:
patch_size = 9
config = patch_results_medium[patch_size]

noisy_gaussian = noise.apply_noise(
    clean_val,
    noise_type="gaussian",
    level="medium",
    noise_config=NOISE_CONFIG,
    seed=SEED
)

denoised_gaussian = reconstruct_image_svr(
    model=config["model"],
    scaler=config["scaler"],
    noisy_img=noisy_gaussian,
    patch_size=patch_size
)

results_gaussian = evaluate_denoising(
    clean_val,
    noisy_gaussian,
    denoised_gaussian
)

print(results_gaussian)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(clean_val, cmap="gray")
plt.title("Clean")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(noisy_gaussian, cmap="gray")
plt.title("Gaussian medium")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(denoised_gaussian, cmap="gray")
plt.title(f"SVR denoised ({patch_size}x{patch_size})")
plt.axis("off")

plt.show()

In [ ]:
# Rician only 

patch_size = 5
config = patch_results_medium[patch_size]

noisy_rician = noise.apply_noise(
    clean_val,
    noise_type="rician",
    level="medium",
    noise_config=NOISE_CONFIG,
    seed=SEED
)

denoised_rician = reconstruct_image_svr(
    model=config["model"],
    scaler=config["scaler"],
    noisy_img=noisy_rician,
    patch_size=patch_size
)

results_rician = evaluate_denoising(
    clean_val,
    noisy_rician,
    denoised_rician
)

print(results_rician)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(clean_val, cmap="gray")
plt.title("Clean")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(noisy_rician, cmap="gray")
plt.title("Rician medium")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(denoised_rician, cmap="gray")
plt.title(f"SVR denoised ({patch_size}x{patch_size})")
plt.axis("off")

plt.show()

### IV. Patch Overlap Study

In [ ]:
# Non-Overlapping patches - stride = patch_size
# Medium Noise only 

PATCH_SIZES = [3, 5, 7, 9]

patch_results_SC = {}

for patch_size in PATCH_SIZES:

    stride = patch_size

    print(
        f"Training non-overlapping | "
        f"patch={patch_size}x{patch_size} | stride={stride}"
    )

    model, scaler = train_svr_configuration(
        train_samples=train_samples,
        noise_types=("gaussian", "rician", "mixed"),
        noise_level="medium",
        patch_size=patch_size,
        stride=stride,
        max_samples=300_000,
        seed=SEED
    )

    patch_results_SC[patch_size] = {
        "model": model,
        "scaler": scaler,
        "stride": stride
    }

In [ ]:
results_SC = []

for patch_size, config in patch_results_SC.items():

    print(
        f"Evaluating non-overlapping | "
        f"{patch_size}x{patch_size} | "
        f"stride={config['stride']}"
    )

    denoised = reconstruct_image_svr(
        model=config["model"],
        scaler=config["scaler"],
        noisy_img=noisy_val_medium,
        patch_size=patch_size
    )

    metrics_result = evaluate_denoising(
        clean_val,
        noisy_val_medium,
        denoised
    )

    metrics_result["patch_size"] = patch_size
    metrics_result["stride"] = config["stride"]
    metrics_result["noise_type"] = noise_types
    metrics_result["noise_level"] = "medium"

    results_SC.append(metrics_result)

df_SC = pd.DataFrame(results_SC)

df_SC

### V. Multi-patient/Multi-noise Evaluation

In [ ]:
# Test Over 3 patients from the test set
# One model/config is retained  : 
#   patch_size = 3, 
#   stride = 2, 
#   noise_types = ("gaussian", "rician", "mixed"), 
#   noise_level = "medium"

final_patch_size = 3

final_config = patch_results_medium[final_patch_size]

final_model = final_config["model"]
final_scaler = final_config["scaler"]

print("Patch size :", final_patch_size)
print("Training stride :", final_config["stride"])

In [ ]:
test_patients = svr_test_files[:3]
slice_number = 0

noise_types_test = [
    "gaussian",
    "rician",
    "mixed"
]

results_per_patient = []


Path("../results/patients").mkdir(
    parents=True,
    exist_ok=True
)

for patient_id, patient_path in enumerate(test_patients):

    volume = utils.lire_nii(patient_path)
    clean_img = volume[slice_number]

    for noise_id, noise_type in enumerate(noise_types_test):

        print(
            f"Patient {patient_path.stem} | "
            f"slice {slice_number} | "
            f"{noise_type}"
        )

        # Medium synthetic noise
        noisy_img = noise.apply_noise(
            clean_img,
            noise_type=noise_type,
            level="medium",
            noise_config=NOISE_CONFIG,
            seed=SEED + patient_id * len(noise_types_test) + noise_id
        )

        # Denoising
        denoised_img = reconstruct_image_svr(
            model=final_model,
            scaler=final_scaler,
            noisy_img=noisy_img,
            patch_size=final_patch_size
        )

        denoised_img = np.clip(
            denoised_img,
            0,
            4095
        ).astype(np.uint16)

        # Metrics
        metrics_result = evaluate_denoising(
            clean_img,
            noisy_img,
            denoised_img
        )

        metrics_result["patient"] = patient_path.stem
        metrics_result["slice"] = slice_number
        metrics_result["noise"] = noise_type
        metrics_result["noise_level"] = "medium"
        metrics_result["patch_size"] = final_patch_size
        metrics_result["stride"] = final_config["stride"]

        results_per_patient.append(metrics_result)


        # Visual comparison
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(clean_img, cmap="gray")
        axes[0].set_title("Clean")

        axes[1].imshow(noisy_img, cmap="gray")
        axes[1].set_title(f"Noisy — {noise_type}")

        axes[2].imshow(denoised_img, cmap="gray")
        axes[2].set_title("SVR denoised")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()

        plt.savefig(
            f"../results/patients/"
            f"{patient_path.stem}_{noise_type}_medium.png",
            dpi=300,
            bbox_inches="tight"
        )



In [ ]:
df_patients = pd.DataFrame(results_per_patient)

df_patients

In [ ]:
df_summary = (
    df_patients
    .groupby("noise")
    .mean(numeric_only=True)
    .reset_index()
)

df_summary

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_patch_results.to_csv(
    RESULTS_DIR / "svr_patch_low.csv",
    index=False
)

df_patch_results_medium.to_csv(
    RESULTS_DIR / "svr_patch_medium.csv",
    index=False
)

df_patch_results_strong.to_csv(
    RESULTS_DIR / "svr_patch_strong.csv",
    index=False
)

df_SC.to_csv(
    RESULTS_DIR / "svr_medium_non_overlap.csv",
    index=False
)


df_patients.to_csv(
    RESULTS_DIR / "svr_final_test_3patients.csv",
    index=False
)

df_summary.to_csv(
    RESULTS_DIR / "svr_final_test_summary.csv",
    index=False
)
